In [1]:
import nltk
from nltk.stem import PorterStemmer,WordNetLemmatizer
import re

nltk.download("wordnet")
nltk.download("omw-1.4")

[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\cvhir\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to
[nltk_data]     C:\Users\cvhir\AppData\Roaming\nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


True

In [9]:

import re
from nltk.stem import PorterStemmer, WordNetLemmatizer
from nltk.util import ngrams

text = "OMG!!! I can't open https://abc.com :)"

text = text.lower()
text = re.sub(r"https?://\S+|www\.\S+", "", text)

text = text.replace("can't", "cannot")
text = text.replace("'re", "are")
text = text.replace("won't", "will not")
text = text.replace("'ve", "have")

text = text.replace(":)", "happy")
text = text.replace(":(", "sad")

tokens = re.findall(r"\b\w+\b", text)

# Stemming
stemmer = PorterStemmer()
stems = [stemmer.stem(word) for word in tokens]

# Lemmatization
lemmatizer = WordNetLemmatizer()
lemma = [lemmatizer.lemmatize(word, pos="v") for word in tokens]

# N-grams
unigrams = list(ngrams(tokens, 1))
bigrams = list(ngrams(tokens, 2))
trigrams = list(ngrams(tokens, 3))

print("Cleaned:", text)
print("Tokens:", tokens)
print("Stems:", stems)
print("Lemmas:", lemma)

print("Unigrams:", unigrams)
print("Bigrams:", bigrams)
print("Trigrams:", trigrams)


Cleaned: omg!!! i cannot open  happy
Tokens: ['omg', 'i', 'cannot', 'open', 'happy']
Stems: ['omg', 'i', 'cannot', 'open', 'happi']
Lemmas: ['omg', 'i', 'cannot', 'open', 'happy']
Unigrams: [('omg',), ('i',), ('cannot',), ('open',), ('happy',)]
Bigrams: [('omg', 'i'), ('i', 'cannot'), ('cannot', 'open'), ('open', 'happy')]
Trigrams: [('omg', 'i', 'cannot'), ('i', 'cannot', 'open'), ('cannot', 'open', 'happy')]


In [3]:
import random
import math
from collections import Counter

In [ ]:
sentences =[
    "i love nlp",
    "i love python",
    "i like nlp",
    "you love python"
]

tokens_list = [
    re.findall(r"\b\w+\b",s.lower())
    for s in sentences
]

bigrams = Counter()
context = Counter()

for tokens in tokens_list:
    tokens = ["<s>"]+tokens+["</s>"]

    for i in range(len(tokens)-1):
        bigrams[(tokens[i],tokens[i+1])] += 1
        context[(tokens[i])] += 1

vocab = set(word for tokens in tokens_list for word in tokens)
vocab.add("</s>")
V=len(vocab)

def bigram_prob(w1,w2):
    return ((bigrams[(w1,w2)]+1)/(context[w1] + V))

def sentence_prob(sentence):
    words = re.findall(r"\b\w+\b",sentence.lower())

    words = ["<s>"]+words+["</s>"]


    prob =1.0

    for i in range(len(words) - 1):
        prob *= bigram_prob(words[i],words[i+1])

    return prob

def perplexity(sentence):

    words = re.findall(r"\b\w+\b",sentence.lower())
    
    words = ["<s>"]+words+["</s>"]

    log_prob = 0.0

    N = len(words)-1

    for i in range(N):
        p = bigram_prob(words[i],words[i+1])
        log_prob += math.log2(p)

    return math.exp(-log_prob/N)

def generate(seed_word, max_words=10):
    random.seed(42)
    word = seed_word.lower()
    result = [word]

    for _ in range(max_words):
        choices = list(vocab)
        probabilities = [
            bigram_prob(word, next_word)
            for next_word in choices
        ]

        word = random.choices(
            choices, weights=probabilities, k=1
        )[0]

        if word == "</s>":
            break

        result.append(word)

    return " ".join(result)

# 9. Display results
print("Vocabulary:", vocab)
print("Bigram counts:", bigrams)

print("P(love | i):", bigram_prob("i", "love"))
print("Sentence probability:",
      sentence_prob("i love nlp"))

print("Perplexity:",
      perplexity("i love nlp"))

print("Generated sentence:",
      generate("i"))
    



vocab :  {'like', 'you', 'love', 'i', '</s>', 'python', 'nlp'}
bigrams :  Counter({('<s>', 'i'): 3, ('i', 'love'): 2, ('nlp', '</s>'): 2, ('love', 'python'): 2, ('python', '</s>'): 2, ('love', 'nlp'): 1, ('i', 'like'): 1, ('like', 'nlp'): 1, ('<s>', 'you'): 1, ('you', 'love'): 1})
P(love | i): 0.3
sentence probability :  0.007272727272727273
perplexity :  5.90513287848402


In [13]:
from collections import Counter
import re

sentence = "I like NLP"

# Split sentence into words
words = re.findall(r"\b\w+\b", sentence.lower())

# Split words into characters
words = [list(word) for word in words]

print("Initial words:", words)

for step in range(5):
    pairs = Counter()

    # Count adjacent pairs
    for word in words:
        for i in range(len(word) - 1):
            pairs[(word[i], word[i + 1])] += 1

    if not pairs:
        break

    # Find the most frequent pair
    a, b = max(pairs, key=pairs.get)
    print("\nMerge:", a, "+", b)

    # Merge the pair
    for word in words:
        i = 0
        while i < len(word) - 1:
            if word[i] == a and word[i + 1] == b:
                word[i] = a + b
                word.pop(i + 1)
            i += 1

    print("Words:", words)

print("\nFinal tokens:", words)

Initial words: [['i'], ['l', 'i', 'k', 'e'], ['n', 'l', 'p']]

Merge: l + i
Words: [['i'], ['li', 'k', 'e'], ['n', 'l', 'p']]

Merge: li + k
Words: [['i'], ['lik', 'e'], ['n', 'l', 'p']]

Merge: lik + e
Words: [['i'], ['like'], ['n', 'l', 'p']]

Merge: n + l
Words: [['i'], ['like'], ['nl', 'p']]

Merge: nl + p
Words: [['i'], ['like'], ['nlp']]

Final tokens: [['i'], ['like'], ['nlp']]


In [ ]:

import nltk

nltk.download("punkt_tab")
nltk.download("averaged_perceptron_tagger_eng")

sentence = "The little boy plays happily"

words = nltk.word_tokenize(sentence)
tags = nltk.pos_tag(words)

print("Words:", words)
print("POS Tags:")

for word, tag in tags:
    print(word, ":", tag)


In [ ]:

import nltk

grammar = nltk.CFG.fromstring("""
S -> NP VP
NP -> PRP
NP -> DT NN
NP -> NP PP
VP -> V NP
VP -> VP PP
PP -> P NP
PRP -> 'I'
DT -> 'the'
NN -> 'man' | 'telescope'
V -> 'saw'
P -> 'with'
""")

parser = nltk.ChartParser(grammar)

sentence = "I saw the man with the telescope"
words = sentence.split()

trees = list(parser.parse(words))

print("Number of parse trees:", len(trees))

for tree in trees:
    tree.pretty_print()


In [14]:
import spacy
from spacy import displacy

nlp = spacy.load("en_core_web_sm")

sentence = "The boy eats an apple."
doc = nlp(sentence)

displacy.render(doc, style="dep", jupyter=True)

In [ ]:

import spacy

nlp = spacy.load("en_core_web_sm")

text = """
The food was delicious.
The service was friendly.
I liked the excellent camera.
"""

doc = nlp(text)

print("Dependencies:")

for token in doc:
    print(token.text, token.dep_, token.head.text)

print("\nAspect-Opinion Pairs:")

pairs = []

for token in doc:
    # Example: food -> delicious
    if token.dep_ == "nsubj" and token.head.pos_ == "ADJ":
        pairs.append((token.text, token.head.text))

    # Example: camera -> excellent
    if token.dep_ == "amod":
        pairs.append((token.head.text, token.text))

print(pairs)


In [ ]:

import nltk
from nltk.wsd import lesk
from nltk.corpus import wordnet as wn

nltk.download("wordnet")
nltk.download("omw-1.4")

sentence = "He deposited money in the bank"
words = nltk.word_tokenize(sentence)

sense = lesk(words, "bank", "n")

print("Sentence:", sentence)

if sense:
    print("Selected sense:", sense.name())
    print("Definition:", sense.definition())
else:
    print("No sense found")


In [ ]:
import nltk
from nltk.corpus import wordnet as wn

nltk.download("wordnet")

word1 = "dog"
word2 = "dog"

syn1 = wn.synsets(word1, pos=wn.NOUN)[0]
syn2 = wn.synsets(word2, pos=wn.NOUN)[0]

print("Synset 1:", syn1.name())
print("Synset 2:", syn2.name())
print("Are they identical?", syn1 == syn2)
print("Similarity:", syn1.wup_similarity(syn2))

In [ ]:

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

documents = [
    "I love machine learning",
    "I love deep learning",
    "I love cricket"
]

query = "machine learning"

# 1. Convert documents into TF-IDF vectors
vectorizer = TfidfVectorizer()
doc_vectors = vectorizer.fit_transform(documents)

# 2. Convert query into a TF-IDF vector
query_vector = vectorizer.transform([query])

# 3. Calculate cosine similarity
scores = cosine_similarity(query_vector, doc_vectors)[0]

# 4. Retrieve top-k documents
k = 2
top_indices = scores.argsort()[::-1][:k]

print("Query:", query)
print("\nSimilarity scores:")

for i, score in enumerate(scores):
    print(documents[i], ":", round(score, 3))

print("\nTop", k, "documents:")

for i in top_indices:
    print(documents[i], ":", round(scores[i], 3))




In [4]:

from sklearn.feature_extraction.text import TfidfVectorizer

sentences = [
    "I love machine learning",
    "I like machine learning",
    "Python is easy to learn"
]

v = TfidfVectorizer()
x = v.fit_transform(sentences)

print("Words:", v.get_feature_names_out())
print("Vectors:\n", x.toarray())

Words: ['easy' 'is' 'learn' 'learning' 'like' 'love' 'machine' 'python' 'to']
Vectors:
 [[0.         0.         0.         0.51785612 0.         0.68091856
  0.51785612 0.         0.        ]
 [0.         0.         0.         0.51785612 0.68091856 0.
  0.51785612 0.         0.        ]
 [0.4472136  0.4472136  0.4472136  0.         0.         0.
  0.         0.4472136  0.4472136 ]]


In [1]:
import nltk
from nltk.tokenize import word_tokenize
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction

nltk.download('punkt_tab')

# Given sentences
reference = "the cat is sitting on the mat"
candidate = "the cat is on the mat"

# Tokenization
ref_tokens = word_tokenize(reference.lower())
cand_tokens = word_tokenize(candidate.lower())

print("Reference Tokens:", ref_tokens)
print("Candidate Tokens:", cand_tokens)

# Calculate BLEU score
score = sentence_bleu(
    [ref_tokens],
    cand_tokens,
    smoothing_function=SmoothingFunction().method1
)

print("BLEU Score:", round(score, 4))

[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\cvhir\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


Reference Tokens: ['the', 'cat', 'is', 'sitting', 'on', 'the', 'mat']
Candidate Tokens: ['the', 'cat', 'is', 'on', 'the', 'mat']
BLEU Score: 0.2876


In [3]:
%pip install "transformers<5.0.0" sentencepiece

Defaulting to user installation because normal site-packages is not writeable
     ---------------------------------------- 0.0/44.0 kB ? eta -:--:--
     --------- ------------------------------ 10.2/44.0 kB ? eta -:--:--
     ----------------- -------------------- 20.5/44.0 kB 165.2 kB/s eta 0:00:01
     ----------------------------------- -- 41.0/44.0 kB 281.8 kB/s eta 0:00:01
     -------------------------------------- 44.0/44.0 kB 270.0 kB/s eta 0:00:00
  Using cached tokenizers-0.22.2-cp39-abi3-win_amd64.whl.metadata (7.4 kB)
   ---------------------------------------- 0.0/12.0 MB ? eta -:--:--
   ---------------------------------------- 0.1/12.0 MB 2.2 MB/s eta 0:00:06
    --------------------------------------- 0.2/12.0 MB 3.0 MB/s eta 0:00:04
   - -------------------------------------- 0.5/12.0 MB 3.9 MB/s eta 0:00:03
   -- ------------------------------------- 0.8/12.0 MB 4.0 MB/s eta 0:00:03
   --- ------------------------------------ 0.9/12.0 MB 3.9 MB/s eta 0:00:03
   --- 

  You can safely remove it manually.

[notice] A new release of pip is available: 24.0 -> 26.2.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [7]:
import sys
import transformers

print("Python:", sys.executable)
print("Transformers version:", transformers.__version__)
print("Transformers location:", transformers.__file__)

Python: c:\Program Files\Python312\python.exe
Transformers version: 5.20.2
Transformers location: C:\Users\cvhir\AppData\Roaming\Python\Python312\site-packages\transformers\__init__.py


In [2]:
import sys
!{sys.executable} -m pip install --user --force-reinstall "transformers==4.46.3" sentencepiece

'c:\Program' is not recognized as an internal or external command,
operable program or batch file.


In [3]:
pip install deep-translator

Defaulting to user installation because normal site-packages is not writeable
   ---------------------------------------- 0.0/42.3 kB ? eta -:--:--
   ----------------------------- ---------- 30.7/42.3 kB 640.0 kB/s eta 0:00:01
   ---------------------------------------- 42.3/42.3 kB 508.6 kB/s eta 0:00:00
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.0 -> 26.2.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [5]:
from deep_translator import GoogleTranslator

sentences = [
    "How are you?",
    "I am a computer science student.",
    "I love machine learning.",
    "Natural language processing is interesting."
]

translator = GoogleTranslator(source="en", target="de")

results = translator.translate_batch(sentences)

for sentence, result in zip(sentences, results):
    print("English:", sentence)
    print("German:", result)

TooManyRequests: Server Error: You made too many requests to the server.According to google, you are allowed to make 5 requests per secondand up to 200k requests per day. You can wait and try again later oryou can try the translate_batch function

In [6]:
import spacy

# Load pretrained NLP model
nlp = spacy.load("en_core_web_sm")

# Given dataset
sentences = [
    "Rohan studies at SSN College in Chennai.",
    "Google was founded in September 1998.",
    "Priya works at Microsoft in India."
]

# Process each sentence
for sentence in sentences:
    doc = nlp(sentence)

    print("\nSentence:", sentence)
    print("Named Entities:")

    for ent in doc.ents:
        print(ent.text, "->", ent.label_)


Sentence: Rohan studies at SSN College in Chennai.
Named Entities:
Rohan -> ORG
SSN College -> ORG
Chennai -> GPE

Sentence: Google was founded in September 1998.
Named Entities:
Google -> ORG
September 1998 -> DATE

Sentence: Priya works at Microsoft in India.
Named Entities:
Microsoft -> ORG
India -> GPE


In [8]:

import nltk
import numpy as np
from nltk.tokenize import word_tokenize

nltk.download('punkt_tab')

sentences = [
    "I love machine learning.",
    "Rohan studies computer science.",
    "Attention is useful in NLP."
]

for sentence in sentences:
    print("\nSentence:", sentence)

    # Tokenization
    words = word_tokenize(sentence)
    print("Tokens:", words)

    # Generate sample vectors
    np.random.seed(1)
    X = np.random.rand(len(words), 3)

    # Query, Key, Value
    Q = X
    K = X
    V = X

    # Attention scores
    scores = Q @ K.T / np.sqrt(K.shape[1])

    # Softmax
    exp_scores = np.exp(scores - np.max(scores, axis=1, keepdims=True))
    weights = exp_scores / exp_scores.sum(axis=1, keepdims=True)

    # Attention output
    output = weights @ V

    print("Attention Scores:\n", np.round(scores, 2))
    print("Attention Weights:\n", np.round(weights, 2))
    print("Attention Output:\n", np.round(output, 2))



Sentence: I love machine learning.
Tokens: ['I', 'love', 'machine', 'learning', '.']
Attention Scores:
 [[0.4  0.13 0.19 0.3  0.41]
 [0.13 0.07 0.08 0.17 0.11]
 [0.19 0.08 0.18 0.3  0.2 ]
 [0.3  0.17 0.3  0.54 0.29]
 [0.41 0.11 0.2  0.29 0.47]]
Attention Weights:
 [[0.22 0.17 0.18 0.2  0.23]
 [0.2  0.19 0.19 0.21 0.2 ]
 [0.2  0.18 0.2  0.22 0.2 ]
 [0.2  0.17 0.19 0.25 0.19]
 [0.22 0.16 0.18 0.2  0.24]]
Attention Output:
 [[0.33 0.53 0.23]
 [0.33 0.51 0.24]
 [0.34 0.51 0.25]
 [0.34 0.51 0.27]
 [0.33 0.54 0.23]]

Sentence: Rohan studies computer science.
Tokens: ['Rohan', 'studies', 'computer', 'science', '.']
Attention Scores:
 [[0.4  0.13 0.19 0.3  0.41]
 [0.13 0.07 0.08 0.17 0.11]
 [0.19 0.08 0.18 0.3  0.2 ]
 [0.3  0.17 0.3  0.54 0.29]
 [0.41 0.11 0.2  0.29 0.47]]
Attention Weights:
 [[0.22 0.17 0.18 0.2  0.23]
 [0.2  0.19 0.19 0.21 0.2 ]
 [0.2  0.18 0.2  0.22 0.2 ]
 [0.2  0.17 0.19 0.25 0.19]
 [0.22 0.16 0.18 0.2  0.24]]
Attention Output:
 [[0.33 0.53 0.23]
 [0.33 0.51 0.24]
 [0.34 

[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\cvhir\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


In [ ]:

import re
import math
from collections import Counter

text = "i love machine learning and i love deep learning"

words = re.findall(r"\b\w+\b", text.lower())

window = 2
pair_counts = Counter()
word_counts = Counter()

for i, word in enumerate(words):
    word_counts[word] += 1

    for j in range(max(0, i - window),
                   min(len(words), i + window + 1)):

        if i != j:
            pair_counts[(word, words[j])] += 1

total_pairs = sum(pair_counts.values())
total_words = sum(word_counts.values())

print("PPMI values:")

for (word, context), count in pair_counts.items():
    p_wc = count / total_pairs
    p_w = word_counts[word] / total_words
    p_c = word_counts[context] / total_words

    pmi = math.log2(p_wc / (p_w * p_c))
    ppmi = max(0, pmi)

    print(word, context, ":", round(ppmi, 3))


In [25]:

from gensim.models import Word2Vec
import re

sentences = [
    "i love machine learning",
    "i love deep learning",
    "machine learning is useful",
    "deep learning is interesting"
]

tokenized = [
    re.findall(r"\b\w+\b", s.lower())
    for s in sentences
]

model = Word2Vec(
    sentences=tokenized,
    vector_size=50,
    window=2,
    min_count=1,
    sg=1,
    seed=42,
    workers=1,
    epochs=100
)

print("Vector for machine:")
print(model.wv["machine"])

print("\nSimilar words to learning:")
print(model.wv.most_similar("learning"))


Vector for machine:
[ 1.4193753e-02 -8.7626409e-03  1.6680082e-02 -8.2666315e-03
 -2.5917459e-03  6.4972541e-03 -1.4993038e-02  2.3017160e-03
  1.3718380e-04  1.1365203e-02  1.9903887e-02  6.6285767e-03
 -3.6502311e-03 -3.6639734e-03 -3.2732557e-03  1.2577106e-02
 -7.1475361e-03 -1.3378090e-02 -6.6509247e-03 -1.9062473e-02
 -1.5779208e-02 -1.6435897e-02  1.0822909e-02  8.9200176e-03
  7.9255654e-03 -1.5547025e-03  8.6985435e-03 -1.3511091e-02
  1.6002260e-02 -2.6367491e-06  1.7487884e-02 -1.3891180e-02
 -1.0778086e-04  7.8550279e-03 -1.8814487e-04 -2.1749658e-03
 -1.3391418e-02 -4.7149067e-03 -1.0428825e-02 -7.9453094e-03
  7.3496448e-03  5.2063460e-03  4.3192571e-03 -5.5385325e-03
  1.8397389e-02 -1.6458666e-02 -6.3162865e-03 -1.5276969e-02
 -6.4539774e-03  1.8441442e-02]

Similar words to learning:
[('interesting', 0.048263490200042725), ('deep', -0.009655913338065147), ('love', -0.02288590557873249), ('i', -0.04924793541431427), ('machine', -0.06404990702867508), ('useful', -0.09761

In [24]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB

texts = [
    "win a free prize",
    "claim your free money",
    "hello how are you",
    "let us meet today",
    "win free cash",
    "see you tomorrow"
]

labels = ["spam", "spam", "normal", "normal", "spam", "normal"]

vectorizer = TfidfVectorizer()
X = vectorizer.fit_transform(texts)
model = MultinomialNB()
model.fit(X, labels)

text = input("Enter a message: ")
x = vectorizer.transform([text])

print("Predicted class:", model.predict(x)[0])

Predicted class: normal


In [3]:
import spacy
nlp = spacy.load("en_core_web_sm")
doc = nlp("Rohan is a computer science student. He studies at SSN College. Rohan loves programming, and he practises it every day. His friend Rahul often studies with him. They work on projects together because they enjoy learning new technologies.")
gender = {"rohan": "m", "rahul": "m", "he": "m", "his": "m","him":"m"}   # hard-coded genders
mentions = list(doc.noun_chunks)                              # mentions = noun chunks (names + pronouns included)
print("Mentions:", [m.text for m in mentions])
chains = {}
for m in mentions:                                            # group by gender, or by head word if no gender
    chains.setdefault(gender.get(m.root.lower_, m.root.lower_), []).append(m.text)
print("Chains:", [c for c in chains.values() if len(c) > 1])

Mentions: ['Rohan', 'a computer science student', 'He', 'SSN College', 'Rohan', 'programming', 'he', 'it', 'His friend', 'Rahul', 'him', 'They', 'projects', 'they', 'new technologies']
Chains: [['Rohan', 'He', 'Rohan', 'he', 'Rahul', 'him'], ['They', 'they']]


In [19]:

import spacy

nlp = spacy.load("en_core_web_sm")

text = text = """
Rohan is a computer science student. 
He studies at SSN College. 
Rohan loves programming, and he practises it every day. 
His friend Rahul often studies with him. 
They work on projects together because they enjoy learning new technologies.
"""

doc = nlp(text)

# 1. Hard-coded gender information
female_names = ["alice", "sara", "mary"]
male_names = ["Rahul","Rohan"]

# 2. Find mentions: names, noun chunks and pronouns
mentions = {}

for ent in doc.ents:
    if ent.label_ == "PERSON":
        mentions[(ent.start, ent.end)] = ent

for chunk in doc.noun_chunks:
    mentions[(chunk.start, chunk.end)] = chunk

for token in doc:
    if token.pos_ == "PRON":
        mentions[(token.i, token.i + 1)] = token.doc[
            token.i:token.i + 1
        ]

mentions = sorted(
    mentions.values(), key=lambda m: m.start
)

# 3. Get gender
def get_gender(mention):
    word = mention.text.lower().replace(".", "").split()[-1]

    if word in female_names or word in ["she", "her", "hers"]:
        return "female"

    if word in male_names or word in ["he", "him", "his"]:
        return "male"

    # Title clues
    text_lower = mention.text.lower()

    if text_lower.startswith(("mr ", "mr.", "mr. ")):
        return "male"

    if text_lower.startswith(("mrs ", "mrs.", "mrs. ")):
        return "female"

    return "unknown"

# 4. Get number
def get_number(mention):
    if mention.root.pos_ == "PRON":
        if mention.root.lower_ in ["they", "them", "we", "us"]:
            return "plural"
        return "singular"

    if mention.root.pos_ == "PROPN":
        return "singular"

    if mention.root.tag_ in ["NNS", "NNPS"]:
        return "plural"

    return "singular"

# 5. Score an earlier mention
def score_pair(previous, current):
    score = 0

    g1 = get_gender(previous)
    g2 = get_gender(current)

    if g1 != "unknown" and g2 != "unknown":
        if g1 == g2:
            score += 2
        else:
            score -= 2

    if get_number(previous) == get_number(current):
        score += 2
    else:
        score -= 2

    if previous.root.lemma_ == current.root.lemma_:
        score += 2

    distance = current.sent.start - previous.sent.start
    score -= max(0, distance // 5)

    return score

# 6. Build coreference chains
chains = []

for i, mention in enumerate(mentions):
    best = None
    best_score = 1

    for previous in mentions[:i]:
        score = score_pair(previous, mention)

        if score > best_score:
            best_score = score
            best = previous

    if best is not None:
        # Add to the chain containing the antecedent
        found = False

        for chain in chains:
            if best in chain:
                chain.append(mention)
                found = True
                break

        if not found:
            chains.append([best, mention])

# 7. Display mentions and chains
print("Mentions:")
for mention in mentions:
    print(mention.text, "->", get_gender(mention),
          ",", get_number(mention))

print("\nCoreference Chains:")
for chain in chains:
    print([m.text for m in chain])


Mentions:

Rohan -> unknown , singular
a computer science student -> unknown , singular
He -> male , singular
SSN College -> unknown , singular
Rohan -> unknown , singular
programming -> unknown , singular
he -> male , singular
it -> unknown , singular
His friend -> unknown , singular
His -> male , singular
Rahul -> unknown , singular
him -> male , singular
They -> unknown , plural
projects -> unknown , plural
they -> unknown , plural
new technologies -> unknown , plural

Coreference Chains:
['\nRohan', 'a computer science student']
['He', 'SSN College', 'he', 'His', 'him']
['Rohan', 'programming', 'it']
['His friend', 'Rahul']
['They', 'projects', 'they', 'new technologies']


In [21]:
import spacy

nlp = spacy.load("en_core_web_sm")

print(nlp.pipe_names)

['tok2vec', 'tagger', 'parser', 'attribute_ruler', 'lemmatizer', 'ner']


In [ ]:

pip install transformers torch

from transformers import pipeline

# Create fill-mask pipeline
fill_mask = pipeline(
    "fill-mask",
    model="bert-base-uncased"
)

text = "I drink a cup of [MASK]."

results = fill_mask(text)

for result in results:
    print("Word:", result["token_str"])
    print("Probability:", result["score"])
    print()


In [ ]:

from transformers import pipeline

# Load sentiment analysis pipeline
sentiment = pipeline("sentiment-analysis")

texts = [
    "I love this movie",
    "This movie is terrible"
]

for text in texts:
    result = sentiment(text)[0]

    print("Text:", text)
    print("Sentiment:", result["label"])
    print("Score:", result["score"])
    print()


In [ ]:

import numpy as np
from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer
)
from sklearn.metrics import accuracy_score, f1_score
from sklearn.model_selection import train_test_split

# 1. Dataset: 9 samples
texts = [
    "I love this movie",
    "This is a wonderful film",
    "The acting was excellent",
    "I really enjoyed it",
    "Amazing and beautiful story",
    "I hate this movie",
    "This film is terrible",
    "The acting was bad",
    "I really disliked it"
]

labels = [1, 1, 1, 1, 1, 0, 0, 0, 0]

# 2. Split data
X_train, X_test, y_train, y_test = train_test_split(
    texts,
    labels,
    test_size=0.33,
    random_state=42,
    stratify=labels
)

# 3. Load tokenizer
model_name = "bert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_name)

# 4. Tokenize the text
train_encodings = tokenizer(
    X_train, truncation=True, padding=True
)
test_encodings = tokenizer(
    X_test, truncation=True, padding=True
)

# 5. Create datasets
train_data = Dataset.from_dict({
    **train_encodings,
    "labels": y_train
})

test_data = Dataset.from_dict({
    **test_encodings,
    "labels": y_test
})

# 6. Load BERT for classification
model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=2
)

# 7. Evaluation metrics
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=1)

    return {
        "accuracy": accuracy_score(labels, predictions),
        "f1": f1_score(labels, predictions, zero_division=0)
    }

# 8. Training settings
args = TrainingArguments(
    output_dir="./results",
    num_train_epochs=2,
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    report_to="none",
    seed=42
)

# 9. Train the model
trainer = Trainer(
    model=model,
    args=args,
    train_dataset=train_data,
    eval_dataset=test_data,
    compute_metrics=compute_metrics
)

trainer.train()

# 10. Evaluate
print(trainer.evaluate())

# 11. Predict new text
new_text = ["The movie was fantastic"]

inputs = tokenizer(
    new_text, return_tensors="pt",
    truncation=True, padding=True
)

import torch

model.eval()

with torch.no_grad():
    outputs = model(**inputs)

predictions = torch.argmax(
    outputs.logits, dim=1
).tolist()

for text, pred in zip(new_text, predictions):
    label = "Positive" if pred == 1 else "Negative"
    print(text, "->", label)


In [28]:

import spacy

nlp = spacy.load("en_core_web_sm")

text = """
Ravi went to the library. He met Priya there.
She helped him find a book. They went home together.
"""

doc = nlp(text)

entities = {}

for ent in doc.ents:
    if ent.label_ == "PERSON":
        entities[ent.text] = []

last_person = None

for token in doc:
    if token.ent_type_ == "PERSON":
        last_person = token.text
        entities.setdefault(last_person, [])
        entities[last_person].append(token.text)

    elif token.pos_ == "PRON":
        if token.lower_ in ["he", "him", "his"]:
            candidates = [
                name for name in entities
                if name.lower() in ["he", "him", "his"]
            ]
        else:
            candidates = []

        if last_person and token.lower_ in ["he", "him", "his"]:
            entities[last_person].append(token.text)

print("Coreference Chains:")
for name, mentions in entities.items():
    print(name, "->", mentions)


Coreference Chains:
Priya -> ['Priya', 'him']


In [5]:
from transformers import MarianMTModel, MarianTokenizer

# Load pretrained translation model (English to French)
model_name = "Helsinki-NLP/opus-mt-en-fr"

tokenizer = MarianTokenizer.from_pretrained(model_name)
model = MarianMTModel.from_pretrained(model_name)

# Input sentence
text = input("Enter English sentence: ")

# Tokenize
tokens = tokenizer(text, return_tensors="pt", padding=True)

# Translate
translated = model.generate(**tokens)

# Decode output
result = tokenizer.decode(translated[0], skip_special_tokens=True)

print("English:", text)
print("French:", result)

KeyboardInterrupt: 

In [7]:
import spacy

# Load spaCy model
nlp = spacy.load("en_core_web_sm")

# Input sentence
text = "Tarun met Mary. He greeted her.Him was good to her"

# Process text
doc = nlp(text)

# Store names and genders
names = {
    "tarun": "M",
    "john": "M",
    "mary": "F"
}

# Store pronouns and genders
pronouns = {
    "he": "M",
    "him": "M",
    "his": "M",
    "she": "F",
    "her": "F",
    "hers": "F"
}

# Remember the latest name for each gender
last_seen = {}

print("Input:", text)
print("\nCOREFERENCE:")

# Identify pronoun references
for token in doc:
    word = token.text.lower()

    if word in names:
        last_seen[names[word]] = token.text

    elif word in pronouns:
        person = last_seen.get(pronouns[word])

        if person:
            print(token.text, "->", person)

Input: Tarun met Mary. He greeted her.Him was good to her

COREFERENCE:
He -> Tarun
her -> Mary
Him -> Tarun
her -> Mary


In [ ]:
import torch
import torch.nn as nn

# 1. Training text
text = "i love nlp i love python nlp is easy python is interesting"

# 2. Create vocabulary
words = text.lower().split()
vocab = list(set(words))

word2idx = {w: i for i, w in enumerate(vocab)}
idx2word = {i: w for w, i in word2idx.items()}

# 3. Create input and output sequences
X, Y = [], []

for i in range(len(words) - 2):
    X.append([word2idx[words[i]], word2idx[words[i+1]]])
    Y.append(word2idx[words[i+2]])

X = torch.tensor(X)
Y = torch.tensor(Y)

# 4. Build neural language model
model = nn.Sequential(
    nn.Embedding(len(vocab), 10),
    nn.Flatten(),
    nn.Linear(20, 32),
    nn.ReLU(),
    nn.Linear(32, len(vocab))
)

# 5. Train the model
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)

for epoch in range(200):
    output = model(X)
    loss = loss_fn(output, Y)

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

print("Training completed!")

# 6. Predict the next word
w1 = input("Enter first word: ").lower()
w2 = input("Enter second word: ").lower()

if w1 in word2idx and w2 in word2idx:
    x = torch.tensor([[word2idx[w1], word2idx[w2]]])
    pred = model(x).argmax(dim=1).item()
    print("Predicted next word:", idx2word[pred])
else:
    print("Word not found in vocabulary!")